# Kuka IIWA 14 Robot Arm Simulator

This notebook demonstrates how to load and simulate the Kuka IIWA 14 robot arm in MuJoCo.

## Import Required Libraries

In [2]:
import numpy as np
import mujoco
import mujoco.viewer
from pathlib import Path
import time
import os

## Load the Kuka IIWA 14 Model

In [3]:
# Path to the Kuka IIWA 14 model
model_path = Path("scene.xml")
if not model_path.exists():
    model_path = Path("iiwa14.xml")

print(f"Loading model from: {model_path}")

# Load the model
model = mujoco.MjModel.from_xml_path(str(model_path))
data = mujoco.MjData(model)

print(f"Model loaded successfully!")
print(f"Number of joints: {model.njnt}")
print(f"Number of actuators: {model.nu}")
print(f"Number of bodies: {model.nbody}")

Loading model from: scene.xml
Model loaded successfully!
Number of joints: 7
Number of actuators: 7
Number of bodies: 9


## Model Information

In [4]:
# Print joint names
print("Joint names:")
for i in range(model.njnt):
    joint_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, i)
    print(f"  {i}: {joint_name}")

print("\nActuator names:")
for i in range(model.nu):
    actuator_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    print(f"  {i}: {actuator_name}")

print("\nBody names:")
for i in range(model.nbody):
    body_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_BODY, i)
    print(f"  {i}: {body_name}")

Joint names:
  0: joint1
  1: joint2
  2: joint3
  3: joint4
  4: joint5
  5: joint6
  6: joint7

Actuator names:
  0: actuator1
  1: actuator2
  2: actuator3
  3: actuator4
  4: actuator5
  5: actuator6
  6: actuator7

Body names:
  0: world
  1: base
  2: link1
  3: link2
  4: link3
  5: link4
  6: link5
  7: link6
  8: link7


## Initialize Simulation

In [5]:
# Reset to initial state (similar to test_mujoco.py)
mujoco.mj_resetData(model, data)

# Set initial joint positions if desired
# Example: Set first joint to 30 degrees (0.5236 radians)
if model.nq > 0:
    data.qpos[0] = 0.5236  # 30 degrees
if model.nq > 1:
    data.qpos[1] = 0.0     # Second joint to 0

# Zero velocities
data.qvel[:] = 0.0

# Recompute positions and forces
mujoco.mj_forward(model, data)

print("Simulation initialized!")
print(f"Initial joint positions: {data.qpos[:model.nq]}")
print(f"Initial joint velocities: {data.qvel[:model.nv]}")
print(f"Control inputs available: {model.nu}")

Simulation initialized!
Initial joint positions: [0.5236 0.     0.     0.     0.     0.     0.    ]
Initial joint velocities: [0. 0. 0. 0. 0. 0. 0.]
Control inputs available: 7


## Interactive Simulation (Main Demo)

## Quick Test (5 seconds only)

Try this first - it's a very short test to verify everything works without risk of crashes.

In [ ]:
# Simple 5-second test with more stable viewer approach
print("Starting 5-second test with stable viewer...")
print("This should open a viewer window and close automatically")

# Reset simulation
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

# Alternative approach: use a manual loop instead of context manager
viewer = None
try:
    # Create viewer without context manager
    viewer = mujoco.viewer.launch_passive(model, data)
    start_time = time.time()
    step = 0
    
    print("Viewer created successfully")
    
    # Very simple 5-second loop
    while step < 500 and (time.time() - start_time) < 5.0:  # Limit both time and steps
        try:
            # Check if viewer is still alive
            if not viewer.is_running():
                print("Viewer closed by user")
                break
                
            # Just let the robot sit there (zero control)
            data.ctrl[:] = 0.0
            
            # Step physics
            mujoco.mj_step(model, data)
            step += 1
            
            # Sync viewer less frequently
            if step % 5 == 0:  # Only sync every 5 steps
                viewer.sync()
            
            # Simple timing - longer sleep
            time.sleep(0.02)
            
        except Exception as e:
            print(f"Error in test loop: {e}")
            break
            
    print("Test loop completed")
    
except Exception as e:
    print(f"Error creating viewer: {e}")

finally:
    # Manual cleanup
    if viewer is not None:
        try:
            print("Closing viewer manually...")
            viewer.close()  # Try explicit close
            print("Viewer closed")
        except:
            print("Viewer close failed, but continuing...")
        viewer = None
    
    # Extra cleanup time
    time.sleep(0.5)
    print("Test finished with manual cleanup")

print("5-second test completed!")

Starting 5-second test...
This should open a viewer window and close automatically
Test completed - viewer should close now...
Test finished.
5-second test completed successfully!


: 

## Backup Test (No Viewer)

If the viewer keeps crashing, try this - it tests the simulation without any graphics.

In [6]:
# Headless simulation test - no viewer, just text output
print("Running headless simulation for 3 seconds...")
print("This tests the physics without any graphics")

# Reset simulation
mujoco.mj_resetData(model, data)

# Set initial pose
if model.nq > 0:
    data.qpos[0] = 0.5  # 30 degrees
data.qvel[:] = 0.0

mujoco.mj_forward(model, data)

print(f"Starting position: {data.qpos[:min(3, model.nq)]}")

# Run simulation for 300 steps (about 3 seconds at 100Hz)
for step in range(300):
    # Simple sinusoidal control
    t = step * model.opt.timestep
    
    if model.nu > 0:
        data.ctrl[0] = 0.3 * np.sin(t * 2)
    if model.nu > 1:  
        data.ctrl[1] = 0.2 * np.cos(t * 1.5)
    
    # Step physics
    mujoco.mj_step(model, data)
    
    # Print every 50 steps
    if step % 50 == 0:
        print(f"Step {step}: t={t:.2f}s, pos={[f'{q:.2f}' for q in data.qpos[:3]]}, ctrl={[f'{c:.2f}' for c in data.ctrl[:2]]}")

print(f"Final position: {data.qpos[:min(3, model.nq)]}")
print("Headless simulation completed successfully!")
print("If this works, the problem is with the viewer, not the simulation.")

Running headless simulation for 3 seconds...
This tests the physics without any graphics
Starting position: [0.5 0.  0. ]
Step 0: t=0.00s, pos=['0.49', '0.00', '0.00'], ctrl=['0.00', '0.20']
Step 50: t=0.10s, pos=['0.20', '0.12', '-0.00'], ctrl=['0.06', '0.20']
Step 100: t=0.20s, pos=['0.13', '0.18', '0.00'], ctrl=['0.12', '0.19']
Step 150: t=0.30s, pos=['0.14', '0.19', '0.00'], ctrl=['0.17', '0.18']
Step 200: t=0.40s, pos=['0.18', '0.19', '0.00'], ctrl=['0.22', '0.17']
Step 250: t=0.50s, pos=['0.22', '0.17', '0.00'], ctrl=['0.25', '0.15']
Final position: [2.49419128e-01 1.52457733e-01 9.50801101e-05]
Headless simulation completed successfully!
If this works, the problem is with the viewer, not the simulation.


## Alternative Viewer Method

If the standard viewer crashes, try this alternative approach.

In [ ]:
# Alternative viewer method - using thread-safe approach
print("Trying alternative viewer method...")

import threading
import sys

# Global flag for stopping simulation
stop_simulation = False

def simulation_loop():
    """Run simulation in a separate function for better isolation"""
    global stop_simulation
    
    # Reset simulation
    mujoco.mj_resetData(model, data)
    mujoco.mj_forward(model, data)
    
    step = 0
    start_time = time.time()
    
    try:
        # Use a simple viewer call without context manager
        viewer = mujoco.viewer.launch_passive(model, data)
        
        print("Viewer launched successfully")
        
        while not stop_simulation and viewer.is_running() and (time.time() - start_time) < 10.0:
            try:
                # Simple control
                data.ctrl[:] = 0.0
                
                # Step physics
                mujoco.mj_step(model, data)
                step += 1
                
                # Update viewer
                viewer.sync()
                
                # Print progress
                if step % 100 == 0:
                    elapsed = time.time() - start_time
                    print(f"Step {step}, Time: {elapsed:.1f}s")
                
                # Sleep
                time.sleep(0.01)
                
            except KeyboardInterrupt:
                print("Interrupted by user")
                break
            except Exception as e:
                print(f"Loop error: {e}")
                break
        
        print("Simulation loop ended")
        
    except Exception as e:
        print(f"Viewer launch failed: {e}")
        return False
    
    return True

# Run the simulation
try:
    print("Starting isolated simulation...")
    success = simulation_loop()
    if success:
        print("Simulation completed successfully")
    else:
        print("Simulation had issues")
except Exception as e:
    print(f"Simulation error: {e}")
finally:
    stop_simulation = True
    print("Cleanup completed")

print("Alternative viewer test finished.")

Trying alternative viewer method...
Starting isolated simulation...
Viewer launched successfully


/home/user/anaconda3/envs/space-robotics/lib/python3.8/site-packages/glfw/__init__.py:917: GLFWError: (65548) b'Wayland: The platform does not provide the window position'
  warnings.warn(message, GLFWError)


Step 100, Time: 2.1s
Step 200, Time: 3.3s
Simulation loop ended
Simulation completed successfully
Cleanup completed
Alternative viewer test finished.


: 

In [ ]:
# Run simulation with viewer (based on test_mujoco.py approach)
print("Starting interactive simulation...")
print("Controls:")
print("- Mouse: rotate view")
print("- Scroll: zoom")
print("- ESC: close")
print("- SPACE: pause/play")
print("- Simulation will auto-stop after 30 seconds to avoid kernel issues")

# Reset to initial state
mujoco.mj_resetData(model, data)

# Set initial condition (optional)
if model.nq > 0:
    data.qpos[0] = 0.5236  # 30 degrees for first joint
    data.qvel[0] = 0.0     # zero velocity

mujoco.mj_forward(model, data)  # recompute positions, forces
print(f"Initial joint position: {float(data.qpos[0]):.3f} radians")

# Run simulation with viewer and proper error handling
start_time = time.time()
max_duration = 30.0  # Auto-stop after 30 seconds

try:
    with mujoco.viewer.launch_passive(model, data) as viewer:
        step = 0
        while viewer.is_running() and (time.time() - start_time) < max_duration:
            try:
                step_start = time.time()

                # Simple control example - set control inputs to zero (free motion)
                data.ctrl[:] = 0.0
                
                # You can add custom control here, for example:
                # data.ctrl[0] = 0.1 * np.sin(step * 0.01)  # sinusoidal control

                # Advance physics
                mujoco.mj_step(model, data)
                step += 1

                # Print joint positions every 100 steps
                if step % 100 == 0:
                    elapsed = time.time() - start_time
                    remaining = max_duration - elapsed
                    print(f"Step {step}: Joint positions: {[f'{q:.3f}' for q in data.qpos[:min(3, model.nq)]]} (Time left: {remaining:.1f}s)")

                # Sync viewer at realtime rate
                viewer.sync()

                # Time keeping to maintain real-time rate
                time_until_next_step = model.opt.timestep - (time.time() - step_start)
                if time_until_next_step > 0:
                    time.sleep(time_until_next_step)
                    
            except Exception as e:
                print(f"Error in simulation loop: {e}")
                break
                
except Exception as e:
    print(f"Viewer error: {e}")
finally:
    # Ensure cleanup
    print("Cleaning up viewer...")
    time.sleep(0.1)  # Small delay to ensure proper cleanup

print("Simulation ended gracefully.")

Starting interactive simulation...
Controls:
- Mouse: rotate view
- Scroll: zoom
- ESC: close
- SPACE: pause/play
- Simulation will auto-stop after 30 seconds to avoid kernel issues
Initial joint position: 0.524 radians


/home/user/anaconda3/envs/space-robotics/lib/python3.8/site-packages/glfw/__init__.py:917: GLFWError: (65548) b'Wayland: The platform does not provide the window position'
  warnings.warn(message, GLFWError)


Step 100: Joint positions: ['0.069', '-0.000', '-0.000'] (Time left: 29.4s)
Step 200: Joint positions: ['0.009', '-0.000', '-0.000'] (Time left: 29.2s)
Step 200: Joint positions: ['0.009', '-0.000', '-0.000'] (Time left: 29.2s)
Step 300: Joint positions: ['0.001', '-0.000', '-0.000'] (Time left: 28.9s)
Step 300: Joint positions: ['0.001', '-0.000', '-0.000'] (Time left: 28.9s)
Step 400: Joint positions: ['0.000', '-0.000', '-0.000'] (Time left: 28.7s)
Step 400: Joint positions: ['0.000', '-0.000', '-0.000'] (Time left: 28.7s)
Step 500: Joint positions: ['0.000', '-0.000', '-0.000'] (Time left: 28.5s)
Step 500: Joint positions: ['0.000', '-0.000', '-0.000'] (Time left: 28.5s)
Step 600: Joint positions: ['0.000', '-0.000', '-0.000'] (Time left: 28.3s)
Step 600: Joint positions: ['0.000', '-0.000', '-0.000'] (Time left: 28.3s)
Step 700: Joint positions: ['0.000', '-0.000', '-0.000'] (Time left: 28.1s)
Step 700: Joint positions: ['0.000', '-0.000', '-0.000'] (Time left: 28.1s)
Step 800: Jo

## Controlled Motion Demo

In [ ]:
# Controlled motion demonstration (based on test_mujoco.py)
print("Starting controlled motion demo...")

# Reset simulation
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

# Run simulation with controlled motion and error handling
try:
    with mujoco.viewer.launch_passive(model, data) as viewer:
        step = 0
        start_time = time.time()
        
        while viewer.is_running() and (time.time() - start_time) < 20:  # Run for 20 seconds
            try:
                sim_start = time.time()

                # Example control: sinusoidal motion on different joints
                t = data.time
                
                if model.nu > 0:
                    data.ctrl[0] = 0.5 * np.sin(t * 2)  # First joint
                if model.nu > 1:
                    data.ctrl[1] = 0.3 * np.sin(t * 1.5 + 0.5)  # Second joint
                if model.nu > 2:
                    data.ctrl[2] = 0.2 * np.sin(t * 1.0)  # Third joint
                if model.nu > 3:
                    data.ctrl[3] = -0.4 * np.sin(t * 0.8)  # Fourth joint

                # Advance physics
                mujoco.mj_step(model, data)
                step += 1

                # Print status every 200 steps
                if step % 200 == 0:
                    print(f"Time: {t:.2f}s, Controls: {[f'{c:.2f}' for c in data.ctrl[:min(4, model.nu)]]}")

                # Sync viewer
                viewer.sync()

                # Maintain real-time rate
                time_until_next_step = model.opt.timestep - (time.time() - sim_start)
                if time_until_next_step > 0:
                    time.sleep(time_until_next_step)
                    
            except Exception as e:
                print(f"Error in controlled motion loop: {e}")
                break
                
except Exception as e:
    print(f"Viewer error in controlled motion: {e}")
finally:
    # Ensure cleanup
    print("Cleaning up controlled motion viewer...")
    time.sleep(0.1)

print("Controlled motion demo completed!")

Starting controlled motion demo...
Time: 0.40s, Controls: ['0.36', '0.27', '0.08', '-0.13']
Time: 0.40s, Controls: ['0.36', '0.27', '0.08', '-0.13']
Time: 0.80s, Controls: ['0.50', '0.30', '0.14', '-0.24']
Time: 0.80s, Controls: ['0.50', '0.30', '0.14', '-0.24']
Time: 1.20s, Controls: ['0.34', '0.22', '0.19', '-0.33']
Time: 1.20s, Controls: ['0.34', '0.22', '0.19', '-0.33']
Time: 1.60s, Controls: ['-0.03', '0.07', '0.20', '-0.38']
Time: 1.60s, Controls: ['-0.03', '0.07', '0.20', '-0.38']
Time: 2.00s, Controls: ['-0.38', '-0.10', '0.18', '-0.40']
Time: 2.00s, Controls: ['-0.38', '-0.10', '0.18', '-0.40']
Time: 2.40s, Controls: ['-0.50', '-0.24', '0.14', '-0.38']
Time: 2.40s, Controls: ['-0.50', '-0.24', '0.14', '-0.38']
Time: 2.80s, Controls: ['-0.32', '-0.30', '0.07', '-0.31']
Time: 2.80s, Controls: ['-0.32', '-0.30', '0.07', '-0.31']
Time: 3.20s, Controls: ['0.06', '-0.25', '-0.01', '-0.22']
Time: 3.20s, Controls: ['0.06', '-0.25', '-0.01', '-0.22']
Time: 3.60s, Controls: ['0.40', '-0

## Custom Control Example

You can customize the robot's behavior by modifying the control inputs.

In [ ]:
# Custom control patterns - modify this cell to experiment!
print("Custom control demo - modify the control logic below!")
print("Simulation will auto-stop after 30 seconds to prevent kernel issues")

# Reset simulation
mujoco.mj_resetData(model, data)

# Set initial pose
if model.nq >= 3:
    data.qpos[0] = 0.0   # Joint 1
    data.qpos[1] = 0.0   # Joint 2  
    data.qpos[2] = 0.0   # Joint 3

mujoco.mj_forward(model, data)

print("Controls available:")
for i in range(model.nu):
    actuator_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    print(f"  ctrl[{i}]: {actuator_name}")

# Interactive simulation with custom control and error handling
start_time = time.time()
max_duration = 30.0  # Auto-stop after 30 seconds

try:
    with mujoco.viewer.launch_passive(model, data) as viewer:
        step = 0
        
        while viewer.is_running() and (time.time() - start_time) < max_duration:
            try:
                step_start = time.time()
                
                # === MODIFY THIS SECTION FOR CUSTOM CONTROL ===
                t = data.time
                
                # Example 1: Step function
                if t < 5:
                    if model.nu > 0: data.ctrl[0] = 0.5  # Move joint 1 for first 5 seconds
                elif t < 10:
                    if model.nu > 1: data.ctrl[1] = -0.3  # Then move joint 2
                else:
                    data.ctrl[:] = 0.0  # Then stop
                    
                # Example 2: Uncomment for wave pattern
                # if model.nu > 0: data.ctrl[0] = 0.3 * np.sin(t)
                # if model.nu > 1: data.ctrl[1] = 0.2 * np.cos(t * 1.5)
                
                # Example 3: Uncomment for position control simulation
                # target_pos = 0.5 * np.sin(t * 0.5)  # Target position
                # if model.nq > 0 and model.nu > 0:
                #     error = target_pos - data.qpos[0]
                #     data.ctrl[0] = 5.0 * error  # Simple P controller
                # === END CUSTOM CONTROL SECTION ===

                # Step simulation
                mujoco.mj_step(model, data)
                step += 1
                
                # Print status
                if step % 300 == 0:
                    elapsed = time.time() - start_time
                    remaining = max_duration - elapsed
                    print(f"t={t:.1f}s: pos={[f'{q:.2f}' for q in data.qpos[:3]]} ctrl={[f'{c:.2f}' for c in data.ctrl[:3]]} (Time left: {remaining:.1f}s)")

                # Sync viewer
                viewer.sync()

                # Maintain timing
                time_until_next_step = model.opt.timestep - (time.time() - step_start)
                if time_until_next_step > 0:
                    time.sleep(time_until_next_step)
                    
            except Exception as e:
                print(f"Error in custom control loop: {e}")
                break
                
except Exception as e:
    print(f"Viewer error in custom control: {e}")
finally:
    # Ensure cleanup
    print("Cleaning up custom control viewer...")
    time.sleep(0.1)

print("Custom control demo ended gracefully.")

Custom control demo - modify the control logic below!
Controls available:
  ctrl[0]: actuator1
  ctrl[1]: actuator2
  ctrl[2]: actuator3
  ctrl[3]: actuator4
  ctrl[4]: actuator5
  ctrl[5]: actuator6
  ctrl[6]: actuator7
t=0.6s: pos=['0.50', '0.00', '0.00'] ctrl=['0.50', '0.00', '0.00']
t=1.2s: pos=['0.50', '-0.00', '0.00'] ctrl=['0.50', '0.00', '0.00']
t=1.8s: pos=['0.50', '-0.00', '-0.00'] ctrl=['0.50', '0.00', '0.00']
t=2.4s: pos=['0.50', '-0.00', '-0.00'] ctrl=['0.50', '0.00', '0.00']
t=3.0s: pos=['0.50', '-0.00', '-0.00'] ctrl=['0.50', '0.00', '0.00']
t=3.6s: pos=['0.50', '-0.00', '-0.00'] ctrl=['0.50', '0.00', '0.00']
t=4.2s: pos=['0.50', '-0.00', '-0.00'] ctrl=['0.50', '0.00', '0.00']
t=4.8s: pos=['0.50', '-0.00', '-0.00'] ctrl=['0.50', '0.00', '0.00']
t=5.4s: pos=['0.50', '-0.31', '-0.00'] ctrl=['0.50', '-0.30', '0.00']
t=6.0s: pos=['0.50', '-0.31', '-0.00'] ctrl=['0.50', '-0.30', '0.00']
t=6.6s: pos=['0.50', '-0.31', '-0.00'] ctrl=['0.50', '-0.30', '0.00']
t=7.2s: pos=['1.08',

## Cleanup

In [ ]:
# Simple data inspection
print("\n=== Final Simulation State ===")
print(f"Final time: {data.time:.2f} seconds")
print(f"Final joint positions: {[f'{q:.3f}' for q in data.qpos[:model.nq]]}")
print(f"Final joint velocities: {[f'{v:.3f}' for v in data.qvel[:model.nv]]}")
print(f"Final control inputs: {[f'{c:.3f}' for c in data.ctrl[:model.nu]]}")
print("\nSimulation complete! The Kuka IIWA 14 model is working properly.")


=== Final Simulation State ===
Final time: 24.11 seconds
Final joint positions: ['0.000', '-0.000', '-0.000', '0.000', '-0.000', '-0.000', '0.000']
Final joint velocities: ['0.000', '0.000', '-0.000', '0.000', '-0.000', '0.000', '-0.000']
Final control inputs: ['0.000', '0.000', '0.000', '0.000', '0.000', '0.000', '0.000']

Simulation complete! The Kuka IIWA 14 model is working properly.


: 